In [1]:
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
print("Client ready!")

Client ready!


In [2]:
with open('../prompts/writer_system_prompt.txt') as f:
    writer_prompt = f.read()
print("Writer prompt loaded!")
print(writer_prompt[:200])

Writer prompt loaded!
You are an expert Technical Product Manager with 10+ years of experience writing Product Requirements Documents (PRDs) for top tech companies.

Your job is to convert rough notes or ideas into a compl


In [3]:
rough_notes = "Build a ride-sharing app for college campuses."

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{writer_prompt}\n\nGenerate a complete PRD from these notes:\n{rough_notes}"
)

prd_output = response.text
print(prd_output)

# Product Requirements Document: CampusRide

## 1. Executive Summary
CampusRide is a specialized, peer-to-peer ride-sharing platform designed exclusively for university and college campuses. Unlike generalized platforms like Uber or Lyft, CampusRide creates a closed-loop ecosystem where drivers and passengers must verify their institutional email (.edu) to join. The application solves the "campus mobility gap"—the lack of affordable, safe, and reliable transportation options for students traveling between dorms, academic buildings, off-campus housing, and local transit hubs. By leveraging shared transit, we reduce campus parking congestion and provide a safer alternative to walking alone at night. This product is for the modern, budget-conscious student who values convenience, campus safety, and peer trust.

---

## 2. Problem Statement
The current transportation landscape on college campuses is fractured. Students face three primary pain points:
1. **Safety Concerns:** Walking across 

In [4]:
with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()
print("Critic prompt loaded!")

Critic prompt loaded!


In [5]:
response2 = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{prd_output}"
)

raw = response2.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 65,
  "missing_sections": [],
  "feedback": [
    "The PRD lacks specific API schema definitions (Endpoints, Methods, Request/Response payloads) in Section 9.",
    "Database schema details are missing; only technologies are listed, lacking an ERD or data model description.",
    "Section 10 is too high-level; lacks a critical path or dependency mapping between dev and regulatory/compliance milestones.",
    "User flow diagrams or wireframe logic are missing, leaving the actual UX design of the 'ride matching' interaction ambiguous.",
    "The GTM strategy mentioned in the note is a fundamental business requirement for a peer-to-peer app and should be a formal section, not an afterthought."
  ],
  "approval_message": "The core concept is sound and well-structured, but the document lacks the technical depth (API/Database design) required for a formal engineering hand-off."
}


In [6]:
# Sirisha's cell - testing critic agent
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()

# Paste Vedha's PRD output here
sample_prd = """# Product Requirements Document (PRD): CampusRide

**Version:** 1.0  
**Status:** Draft  
**Product Manager:** [Your Name]

---

### 1. Executive Summary
CampusRide is a hyper-local, peer-to-peer ride-sharing platform designed exclusively for university students. By verifying user identities via university email domains (.edu), CampusRide fosters a safe, affordable, and sustainable transportation network. The app allows students to offer rides to classmates or request trips to off-campus housing, local transit hubs, and retail centers.

### 2. Problem Statement
College students face significant transportation challenges: limited campus shuttle hours, unreliable public transit, and high costs associated with traditional ride-sharing apps (Uber/Lyft). Furthermore, many students own cars but struggle to cover parking and gas costs, while others struggle with "transportation deserts" surrounding campus. There is currently no platform that provides a trusted, community-based solution that prioritizes safety and university-specific affiliation.

### 3. User Personas
*   **Persona A: "The Commuter" (Sarah)**
    *   *Profile:* A sophomore living 3 miles off-campus. Does not have a car.
    *   *Goal:* Needs a reliable, low-cost way to get to 8:00 AM lectures when the shuttle is full.
*   **Persona B: "The Driver" (David)**
    *   *Profile:* A senior who drives to campus every day.
    *   *Goal:* Wants to offset the cost of high campus parking permits and gas by picking up passengers along his daily route.

### 4. User Stories
1.  **As a user**, I want to sign up with my .edu email address so that I know everyone in the app is a fellow student.
2.  **As a driver**, I want to post my daily route so I can find riders who live near me.
...
*   **Milestone 4 (Month 4):** Beta launch and integration of payment processing; official campus rollout.


TOTAL LENGTH: 5323 characters"""

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{sample_prd}"
)

raw = response.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 42,
  "missing_sections": [
    "5. Functional Requirements",
    "6. Non-Functional Requirements",
    "7. UX/UI Requirements",
    "8. Analytics and Success Metrics",
    "9. Risk Management and Mitigation",
    "10. Go-to-Market Strategy"
  ],
  "feedback": [
    "PRD is incomplete; missing 60% of the required structural sections.",
    "User Stories section is truncated with an ellipsis, indicating a lack of thorough requirements gathering.",
    "Missing critical technical specifications regarding data security, privacy (GDPR/FERPA), and payment infrastructure.",
    "Lack of success metrics (KPIs) makes the project goals unverifiable.",
    "Risk assessment is absent, which is unacceptable for a peer-to-peer transportation platform involving physical safety."
  ],
  "approval_message": "The PRD is woefully incomplete. You have provided a concept overview but failed to define the actual technical requirements, success metrics, or risk fra